In [59]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from scipy.stats.mstats import winsorize

Load raw data

In [60]:
train = pd.read_csv("./Material/train_data.csv")
test  = pd.read_csv("./Material/test_data.csv")

Preprocessing function

In [61]:
fit_values = {}

def preprocess(df, is_train=True):
    df = df.copy()
    df = df.drop(columns=['id'])

    # only drop duplicates from training so we don't lose evaluation rows
    if is_train:
        df = df.drop_duplicates()

    # Handle garbage values
    unseen_work = ["children", "Never_worked"]
    df["work_type"] = df["work_type"].replace(unseen_work, "Unknown")

    # Compute and store imputation values from training data
    if is_train:
        fit_values["age_median"]   = df["Age"].median()
        fit_values["gender_mode"]  = df["Gender"].mode()[0]

    # Impute nulls
    df["Age"] = df["Age"].fillna(round(fit_values["age_median"]))
    df["Gender"] = df["Gender"].fillna(fit_values["gender_mode"])
    df["work_type"] = df["work_type"].fillna("Unknown")
    df["smoking_status"] = df["smoking_status"].fillna("Unknown")

    # Age => median
    # Gender => mode
    # work_type => 'Unknown' (likely structured missingness so we don't want to assert a category)
    # smoking_status => 'Unknown' (consistent with the deliberate 'Unknown' entries already in the data)


    # Cast Age back to int (median imputation makes it float)
    df["Age"] = df["Age"].astype(int)

    # Encode target
    df["Heart Disease"] = df["Heart Disease"].map({"Yes": 1, "No": 0})

    # Encode binary categorical columns
    df["Gender"] = df["Gender"].map({"Female": 0, "Male": 1})

    # One-hot encode nominal categoricals
    # label encoding would imply a false ranking
    df = pd.get_dummies(df, columns=["work_type", "smoking_status"], drop_first=False)

    bool_cols = list(df.select_dtypes(bool).columns)
    df[bool_cols] = df[bool_cols].astype(int)



    # outliers in BP and Cholesterol could be meaningful
    # (e.g. very high BP might indicate severe hypertension)
    # so we'll leave them as-is for the model to learn from
    # except for one extreme outlier in Cholesterol that seems like a clear data error
    # (value of 564, which is 6.04 stds above the mean in training data)
    # so we'll winsorize the Cholesterol column to 99%
    # to prevent it from skewing the scale of the feature

    if is_train:
        df['Cholesterol'] = winsorize(df['Cholesterol'], limits=[0.01, 0.01]).data
        fit_values['cholesterol_min'] = df['Cholesterol'].min()
        fit_values['cholesterol_max'] = df['Cholesterol'].max()

        df['BP'] = winsorize(df['BP'], limits=[0.01, 0.01]).data
        fit_values['bp_min'] = df['BP'].min()
        fit_values['bp_max'] = df['BP'].max()

    df['Cholesterol'] = df['Cholesterol'].clip(
        lower=fit_values['cholesterol_min'],
        upper=fit_values['cholesterol_max']
    )
    df['BP'] = df['BP'].clip(
        lower=fit_values['bp_min'],
        upper=fit_values['bp_max']
    )
    return df


Apply

In [62]:
# Train first so fit_values gets populated
train_clean = preprocess(train, is_train=True)
test_clean  = preprocess(test,  is_train=False)

test_clean = test_clean.drop_duplicates()

Align columns

In [63]:
# after one-hot encoding train and test might have different columns
# if a category only appears in one of them this ensures they match exactly
train_cols = set(train_clean.columns)
test_cols  = set(test_clean.columns)

for col in train_cols - test_cols:
    test_clean[col] = 0  # category exists in train but not test

for col in test_cols - train_cols:
    train_clean[col] = 0  # category exists in test but not train

# Reorder test columns to match train exactly
test_clean = test_clean[train_clean.columns]

Split features and target

In [64]:
x_train = train_clean.drop(columns=["Heart Disease"])
y_train = train_clean["Heart Disease"]

x_test  = test_clean.drop(columns=["Heart Disease"])
y_test  = test_clean["Heart Disease"]

Scale numeric features

In [65]:
# Fit scaler on training data only to avoid data leakage
numeric_cols = ["Age", "BP", "Cholesterol", "Max HR", "ST depression"]

scaler = StandardScaler()
x_train[numeric_cols] = scaler.fit_transform(x_train[numeric_cols])
x_test[numeric_cols]  = scaler.transform(x_test[numeric_cols])

Sanity checks

In [66]:
print("Train shape:", x_train.shape)
print("Test shape: ", x_test.shape)
print("\nNulls in train?", x_train.isnull().any().any())
print("Nulls in test? ", x_test.isnull().any().any())
print("\nColumns match?", list(x_train.columns) == list(x_test.columns))
print("\nTarget distribution (train):")
print(y_train.value_counts())
print("\nTarget distribution (test):")
print(y_test.value_counts())

Train shape: (215, 21)
Test shape:  (55, 21)

Nulls in train? False
Nulls in test?  False

Columns match? True

Target distribution (train):
Heart Disease
0    119
1     96
Name: count, dtype: int64

Target distribution (test):
Heart Disease
0    31
1    24
Name: count, dtype: int64


Export

In [67]:
x_train.to_csv("./Data/Dataset/x_train.csv", index=False)
x_test.to_csv("./Data/Dataset/x_test.csv", index=False)
y_train.to_csv("./Data/Dataset/y_train.csv", index=False)
y_test.to_csv("./Data/Dataset/y_test.csv", index=False)